# Part VIII — Final Models and SHAP Analysis

**Standalone notebook addition.** Run from the repository root in VS Code/Jupyter. This notebook contains only the new final-model analysis, not the earlier EDA.

**Workflow:** rolling-season model selection → train on 2014–15 through 2024–25 → evaluate 2025–26 → global SHAP → local SHAP for selected matches.

**Note:** Club Only is retained as a reference, while the final explanation models are selected from feature sets containing player information. The 2025–26 test season has previously been inspected and is not a fully untouched blind test.

## 1. Imports and feature definitions

In [ ]:
import json
import warnings
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from catboost import CatBoostClassifier, CatBoostRegressor
from scipy.stats import norm, skellam
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, PoissonRegressor, Ridge
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
warnings.filterwarnings('ignore', category=FutureWarning)
ORDER = ['A', 'D', 'H']
POSITION_SLOTS = {
    'GK': ['gk1'],
    'DEF': [f'def{i}' for i in range(1, 5)],
    'MID': [f'mid{i}' for i in range(1, 4)],
    'ATT': [f'att{i}' for i in range(1, 4)]
}
POSITION_ATTRS = {
    'GK': ['overall', 'goalkeeping_kicking', 'goalkeeping_positioning', 'age'],
    'DEF': ['overall', 'potential', 'movement_reactions', 'mentality_interceptions',
            'defending_standing_tackle', 'attacking_short_passing'],
    'MID': ['overall', 'potential', 'movement_reactions', 'attacking_short_passing'],
    'ATT': ['overall', 'potential', 'movement_reactions', 'attacking_finishing']
}
EXTRA_STATS = {
    'DEF': {'mentality_interceptions': 'min', 'defending_standing_tackle': 'min'},
    'MID': {'movement_reactions': 'min'},
    'ATT': {'movement_reactions': 'max', 'attacking_finishing': 'max'}
}
CASE_REQUESTS = [
    ('Real Madrid vs Arsenal', '2024-25', '2025-04-16', 'Real Madrid', 'Arsenal'),
    ('Aston Villa vs PSG', '2024-25', '2025-04-15', 'Aston Villa', 'Paris Saint-Germain'),
    ('Inter vs Bodo/Glimt', '2025-26', '2026-02-24', 'Internazionale', 'Bod'),
    ('Bayern vs Real Madrid', '2025-26', '2026-04-15', 'Bayern', 'Real Madrid'),
    ('PSG vs Arsenal', '2025-26', '2026-05-30', 'Paris Saint-Germain', 'Arsenal')
]


In [ ]:
def club_columns(data):
    attrs = ['overall', 'attack', 'midfield', 'defence', 'domestic_prestige', 'international_prestige']
    cols = [f'{side}_club_{attr}' for side in ['home', 'away'] for attr in attrs]
    cols += ['neutral_venue', 'stage_league', 'stage_knockout']
    missing = [col for col in cols if col not in data.columns]
    if missing:
        raise KeyError(f'Missing Club features: {missing}')
    return cols


def build_position_features(data):
    out = {}
    for side in ['home', 'away']:
        for position, slots in POSITION_SLOTS.items():
            for attr in POSITION_ATTRS[position]:
                cols = [f'{side}_{slot}_{attr}' for slot in slots]
                missing = [col for col in cols if col not in data.columns]
                if missing:
                    raise KeyError(f'Missing Player features: {missing}')

                values = data[cols].apply(pd.to_numeric, errors='coerce')
                key = f'{side}_{position.lower()}_{attr}'
                out[f'{key}_mean'] = values.mean(axis=1)

                extra = EXTRA_STATS.get(position, {}).get(attr)
                if extra == 'min':
                    out[f'{key}_min'] = values.min(axis=1)
                elif extra == 'max':
                    out[f'{key}_max'] = values.max(axis=1)

    return pd.DataFrame(out, index=data.index)


def feature_groups(data):
    club = data[club_columns(data)].apply(pd.to_numeric, errors='coerce')
    position = build_position_features(data)
    defence = position.filter(regex=r'^(home|away)_def_')

    # Club+DEF, Club+Position Core and prior Compact player aggregates.
    compact_cols = [col for col in data if col.startswith(
        ('home_xi_', 'away_xi_', 'home_bench_', 'away_bench_')
    )]
    compact = data[compact_cols].apply(pd.to_numeric, errors='coerce')

    return {
        'club_only': club,
        'club_def': pd.concat([club, defence], axis=1),
        'club_position_core': pd.concat([club, position], axis=1),
        'club_compact': pd.concat([club, compact], axis=1)
    }


## 2. Model definitions and common evaluation

Each regression route is converted to W/D/L probabilities. Draw-decision thresholds are chosen only on historical rolling predictions.

In [ ]:
def make_estimator(route, algorithm, quick=False):
    if route == 'WDL':
        if algorithm == 'Logistic':
            model = LogisticRegression(C=0.1, max_iter=2500, random_state=42)
        elif algorithm == 'XGBoost':
            model = XGBClassifier(n_estimators=90 if quick else 180, max_depth=3,
                learning_rate=0.04, subsample=0.8, colsample_bytree=0.8,
                reg_lambda=10, objective='multi:softprob', num_class=3,
                n_jobs=4, random_state=42)
        elif algorithm == 'CatBoost':
            model = CatBoostClassifier(iterations=100 if quick else 250, depth=4,
                learning_rate=0.04, l2_leaf_reg=10, verbose=False,
                thread_count=4, random_seed=42, loss_function='MultiClass')
        else:
            raise ValueError(algorithm)
    elif route == 'Goal Difference':
        if algorithm == 'Ridge':
            model = Ridge(alpha=10)
        elif algorithm == 'RandomForest':
            model = RandomForestRegressor(n_estimators=100 if quick else 200,
                max_depth=5, min_samples_leaf=8, random_state=42, n_jobs=-1)
        elif algorithm == 'CatBoost':
            model = CatBoostRegressor(iterations=100 if quick else 250, depth=4,
                learning_rate=0.04, l2_leaf_reg=10, verbose=False,
                thread_count=4, random_seed=42)
        else:
            raise ValueError(algorithm)
    else:
        if algorithm == 'Poisson':
            model = PoissonRegressor(alpha=1.0, max_iter=1500)
        elif algorithm == 'RandomForest':
            model = RandomForestRegressor(n_estimators=100 if quick else 200,
                max_depth=5, min_samples_leaf=8, random_state=42, n_jobs=-1)
        elif algorithm == 'CatBoost':
            model = CatBoostRegressor(iterations=100 if quick else 250, depth=4,
                learning_rate=0.04, l2_leaf_reg=10, verbose=False,
                thread_count=4, random_seed=42)
        else:
            raise ValueError(algorithm)

    steps = [SimpleImputer(strategy='median')]
    if algorithm in ('Logistic', 'Ridge', 'Poisson'):
        steps.append(StandardScaler())
    steps.append(model)
    return make_pipeline(*steps)

ALGORITHMS = {
    'WDL': ['Logistic', 'XGBoost', 'CatBoost'],
    'Goal Difference': ['Ridge', 'RandomForest', 'CatBoost'],
    'Two Goals': ['Poisson', 'RandomForest', 'CatBoost']
}


In [ ]:
def fit_bundle(route, algo, X, data, quick=False):
    if route == 'Two Goals':
        models = {}
        for target, col in [('home', 'label_90min_home_goals'), ('away', 'label_90min_away_goals')]:
            model = make_estimator(route, algo, quick)
            model.fit(X, data[col].to_numpy(dtype=float))
            models[target] = model
        return {'route': route, 'algo': algo, 'models': models}

    model = make_estimator(route, algo, quick)
    if route == 'WDL':
        y = data['label_90min_wdl'].map({'A': 0, 'D': 1, 'H': 2})
    else:
        y = data['label_90min_goal_diff'].astype(float)
    model.fit(X, y)
    bundle = {'route': route, 'algo': algo, 'models': {'main': model}}

    if route == 'Goal Difference':
        residual = y.to_numpy() - model.predict(X)
        bundle['sigma'] = max(float(np.std(residual, ddof=1)), 0.5)
    return bundle


def predict_bundle(bundle, X):
    route, models = bundle['route'], bundle['models']
    if route == 'WDL':
        p = models['main'].predict_proba(X)
        # All folds must include each of A, D, H.
        indices = [list(models['main'][-1].classes_).index(v) for v in [0, 1, 2]]
        return p[:, indices], None
    if route == 'Goal Difference':
        mu = models['main'].predict(X)
        sigma = bundle['sigma']
        p_away = norm.cdf(-0.5, loc=mu, scale=sigma)
        p_home = norm.sf(0.5, loc=mu, scale=sigma)
        return np.column_stack([p_away, 1 - p_away - p_home, p_home]), mu

    home = np.clip(models['home'].predict(X), 0.01, 30)
    away = np.clip(models['away'].predict(X), 0.01, 30)
    p_away = skellam.cdf(-1, home, away)
    p_draw = skellam.pmf(0, home, away)
    return np.column_stack([p_away, p_draw, 1 - p_away - p_draw]), (home, away)


def evaluate(y, probs, hard=None):
    probs = np.clip(probs, 1e-9, 1)
    probs = probs / probs.sum(axis=1, keepdims=True)
    if hard is None:
        hard = np.array(ORDER)[probs.argmax(axis=1)]

    metrics = {
        'accuracy': accuracy_score(y, hard),
        'macro_f1': f1_score(y, hard, labels=ORDER, average='macro', zero_division=0),
        'log_loss': log_loss(y, probs, labels=ORDER),
    }
    try:
        metrics['macro_auc'] = roc_auc_score(y, probs, labels=ORDER, multi_class='ovr', average='macro')
    except ValueError:
        metrics['macro_auc'] = np.nan
    return metrics


def decision_labels(route, p, raw, threshold):
    if route == 'WDL':
        return np.array(ORDER)[p.argmax(axis=1)]
    if route == 'Goal Difference':
        return np.where(raw > threshold, 'H', np.where(raw < -threshold, 'A', 'D'))
    return np.where(p[:, 1] >= threshold, 'D', np.where(p[:, 2] >= p[:, 0], 'H', 'A'))


def choose_threshold(route, probs, raw, y):
    if route == 'WDL':
        return None

    grid = np.arange(0, 1.01, 0.05) if route == 'Goal Difference' else np.arange(0.10, 0.401, 0.01)
    candidates = []
    for threshold in grid:
        pred = decision_labels(route, probs, raw, float(threshold))
        acc = accuracy_score(y, pred)
        f1 = f1_score(y, pred, labels=ORDER, average='macro', zero_division=0)
        candidates.append((0.75 * acc + 0.25 * f1, acc, f1, float(threshold)))
    return max(candidates)[-1]


def clean_frames(Xtr, Xva):
    Xtr = Xtr.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    Xva = Xva.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    keep = [c for c in Xtr if Xtr[c].notna().any() and Xtr[c].nunique(dropna=True) > 1]
    if not keep:
        raise ValueError('No non-constant training features')
    return Xtr[keep], Xva[keep], keep


## 3. Load data and choose final candidates

Set `QUICK_RUN = True` for a small check; use `False` for all planned rolling folds and candidate models.

In [ ]:
DATA_PATH = Path('data/match_dataset/matches_full.csv')
OUTPUT = Path('model/results/final_player_shap')
QUICK_RUN = False
RUN_SHAP = True

OUTPUT.mkdir(parents=True, exist_ok=True)
data = pd.read_csv(DATA_PATH, low_memory=False).reset_index(drop=True)

required = ['season', 'match_date', 'home_team', 'away_team', 'label_90min_wdl',
            'label_90min_goal_diff', 'label_90min_home_goals', 'label_90min_away_goals']
missing = [col for col in required if col not in data.columns]
if missing:
    raise KeyError(f'Missing dataset columns: {missing}')

groups = feature_groups(data)
print('Matches:', len(data))
print('Feature dimensions:', {key: len(value.columns) for key, value in groups.items()})

In [ ]:
def rolling_select(data, groups, output, quick=False):
    season = data['season'].str[:4].astype(int)
    folds = [2023, 2024] if quick else [2021, 2022, 2023, 2024]
    variants = ['club_def', 'club_position_core'] if quick else [
        'club_only', 'club_def', 'club_position_core', 'club_compact'
    ]
    records = []
    oof = {}

    for holdout in folds:
        tr, va = season < holdout, season == holdout
        for variant in variants:
            Xtr, Xva, columns = clean_frames(groups[variant].loc[tr], groups[variant].loc[va])
            for route, algorithms in ALGORITHMS.items():
                if quick:
                    algorithms = algorithms[:1]
                for algo in algorithms:
                    key = (route, variant, algo)
                    print(f'Rolling {holdout}: {route} / {variant} / {algo}', flush=True)
                    bundle = fit_bundle(route, algo, Xtr, data.loc[tr], quick)
                    p, raw = predict_bundle(bundle, Xva)
                    y = data.loc[va, 'label_90min_wdl'].to_numpy()
                    metrics = evaluate(y, p)

                    records.append({'season': holdout, 'route': route, 'feature_set': variant,
                                    'model': algo, 'n_features': len(columns), **metrics})
                    oof.setdefault(key, []).append((y, p, raw))

    rolling = pd.DataFrame(records)
    rolling.to_csv(output / 'rolling_all.csv', index=False)
    summary = rolling.groupby(['route', 'feature_set', 'model'], as_index=False)[
        ['accuracy', 'macro_auc', 'macro_f1', 'log_loss']
    ].mean()

    # Select based on accuracy and multiclass ROC-AUC, equally weighted ranks.
    # Club Only is kept as a benchmark, not eligible as the final explained player model.
    eligible = summary[summary.feature_set != 'club_only'].copy()
    eligible['acc_rank'] = eligible.groupby('route')['accuracy'].rank(ascending=False)
    eligible['auc_rank'] = eligible.groupby('route')['macro_auc'].rank(ascending=False)
    eligible['selection_rank'] = (eligible.acc_rank + eligible.auc_rank) / 2
    eligible = eligible.sort_values(['route', 'selection_rank', 'macro_auc', 'accuracy'],
                                    ascending=[True, True, False, False])
    eligible.to_csv(output / 'rolling_candidate_ranking.csv', index=False)
    winners = eligible.drop_duplicates('route').copy()

    threshold_rows = []
    for _, row in winners.iterrows():
        key = (row.route, row.feature_set, row.model)
        parts = oof[key]
        y = np.concatenate([item[0] for item in parts])
        p = np.concatenate([item[1] for item in parts])
        raw = None
        if row.route == 'Goal Difference':
            raw = np.concatenate([item[2] for item in parts])
        if row.route == 'Two Goals':
            raw = None
        threshold = choose_threshold(row.route, p, raw, y)
        threshold_rows.append({'route': row.route, 'threshold': threshold})
    winners = winners.merge(pd.DataFrame(threshold_rows), on='route', how='left')
    winners.to_csv(output / 'selected_models.csv', index=False)

    print('\nSelected models from rolling seasons:')
    print(winners[['route', 'feature_set', 'model', 'accuracy', 'macro_auc', 'threshold']].to_string(index=False))
    return winners, summary


In [ ]:
winners, rolling_summary = rolling_select(data, groups, OUTPUT, quick=QUICK_RUN)
display(winners[['route', 'feature_set', 'model', 'accuracy', 'macro_auc', 'threshold']].round(4))

## 4. Refit and evaluate on the 2025–26 test season

Models are refit on all seasons before 2025–26. The selected thresholds are not tuned again on test data.

In [ ]:
def refit_and_test(data, groups, winners, output, quick=False):
    tr = data.season != '2025-26'
    te = data.season == '2025-26'
    results = []
    saved = {}

    for _, row in winners.iterrows():
        route, variant, algo = row.route, row.feature_set, row.model
        Xtr, Xte, columns = clean_frames(groups[variant].loc[tr], groups[variant].loc[te])
        bundle = fit_bundle(route, algo, Xtr, data.loc[tr], quick)
        bundle.update({'columns': columns, 'feature_set': variant, 'threshold': row.threshold})
        probs, raw = predict_bundle(bundle, Xte)
        threshold = row.threshold if pd.notna(row.threshold) else None
        labels = decision_labels(route, probs, raw, threshold)
        y = data.loc[te, 'label_90min_wdl'].to_numpy()
        metrics = evaluate(y, probs, labels)
        results.append({'route': route, 'feature_set': variant, 'model': algo,
                        'n_features': len(columns), 'threshold': threshold, **metrics})
        saved[route] = bundle

        pred = data.loc[te, ['match_id', 'season', 'match_date', 'home_team', 'away_team',
                            'label_90min_wdl', 'label_90min_home_goals', 'label_90min_away_goals']].copy()
        pred['pred_wdl'] = labels
        for i, c in enumerate(ORDER):
            pred[f'prob_{c}'] = probs[:, i]
        if route == 'Goal Difference':
            pred['pred_goal_diff'] = raw
        elif route == 'Two Goals':
            pred['pred_home_goals'], pred['pred_away_goals'] = raw
        pred.to_csv(output / f'test_predictions_{route.lower().replace(" ", "_")}.csv', index=False)
        joblib.dump(bundle, output / f'model_{route.lower().replace(" ", "_")}.joblib')

    pd.DataFrame(results).to_csv(output / 'final_test_metrics.csv', index=False)
    print('\nFinal held-out 2025-26 metrics:')
    print(pd.DataFrame(results).round(4).to_string(index=False))
    return saved


In [ ]:
saved = refit_and_test(data, groups, winners, OUTPUT, quick=QUICK_RUN)
display(pd.read_csv(OUTPUT / 'final_test_metrics.csv').round(4))

## 5. Global SHAP

Interpret each output in its original scale: classification margins, predicted goal difference, or log expected goals. SHAP values are model explanations, not the real-world cause of a result.

In [ ]:
def numeric_input(bundle, raw):
    X = raw.reindex(columns=bundle['columns']).apply(pd.to_numeric, errors='coerce')
    return X.replace([np.inf, -np.inf], np.nan)


def shap_for_pipeline(pipeline, raw_X, background_X):
    """Explain the underlying estimator in preprocessed feature units."""
    transformer = pipeline[:-1]
    model = pipeline[-1]
    x = transformer.transform(raw_X)
    bg = transformer.transform(background_X)
    columns = raw_X.columns.tolist()

    # LinearExplainer returns model margins: WDL logits, goal difference,
    # or log expected goals for PoissonRegressor.
    if isinstance(model, (LogisticRegression, Ridge, PoissonRegressor)):
        explainer = shap.LinearExplainer(model, bg)
    else:
        explainer = shap.TreeExplainer(model)
    values = explainer(x)
    values.feature_names = columns
    values.data = raw_X.to_numpy()
    return values


def global_shap(data, groups, saved, out, sample_size=120):
    out.mkdir(parents=True, exist_ok=True)
    train = data.season != '2025-26'
    test = data.season == '2025-26'
    rng = np.random.default_rng(42)
    all_rows = []
    expl = {}

    for route, bundle in saved.items():
        raw = numeric_input(bundle, groups[bundle['feature_set']])
        tr = raw.loc[train]
        te = raw.loc[test]
        bg = tr.sample(n=min(100, len(tr)), random_state=42)
        sample = te.sample(n=min(sample_size, len(te)), random_state=42)
        models = bundle['models']
        output_names = ['Home goals', 'Away goals'] if route == 'Two Goals' else ['WDL' if route == 'WDL' else 'Goal Difference']
        explainers = [models['home'], models['away']] if route == 'Two Goals' else [models['main']]

        for name, model in zip(output_names, explainers):
            try:
                sv = shap_for_pipeline(model, sample, bg)
            except Exception as e:
                print(f'SHAP failed: {route} / {name}: {e}')
                continue

            expl[(route, name)] = (sv, sample)
            arr = np.asarray(sv.values)
            if arr.ndim == 3:
                # SHAP for A/D/H classes, class-specific outputs are preferable to averaging them away.
                for j, class_name in enumerate(ORDER):
                    value = np.abs(arr[:, :, j]).mean(axis=0)
                    all_rows += [{'route': route, 'output': f'{name} / {class_name}', 'feature': f,
                                  'mean_abs_shap': float(v)} for f, v in zip(sample.columns, value)]
                    plt.figure(figsize=(9, 6))
                    shap.plots.bar(sv[:, :, j], max_display=20, show=False)
                    plt.title(f'{route} | {class_name} global SHAP')
                    plt.tight_layout()
                    plt.savefig(out / f'global_{route}_{class_name}.png', dpi=180, bbox_inches='tight')
                    plt.close()
            else:
                value = np.abs(arr).mean(axis=0)
                all_rows += [{'route': route, 'output': name, 'feature': f,
                              'mean_abs_shap': float(v)} for f, v in zip(sample.columns, value)]
                plt.figure(figsize=(9, 6))
                shap.plots.bar(sv, max_display=20, show=False)
                plt.title(f'{route} | {name} global SHAP')
                plt.tight_layout()
                plt.savefig(out / f'global_{route}_{name}.png', dpi=180, bbox_inches='tight')
                plt.close()

    importance = pd.DataFrame(all_rows)
    if not importance.empty:
        importance.to_csv(out / 'global_shap_all_features.csv', index=False)
        importance['group'] = importance.feature.apply(feature_group)
        importance.groupby(['route', 'output', 'group'], as_index=False).mean_abs_shap.sum().to_csv(
            out / 'global_shap_position_groups.csv', index=False)
    return importance


def feature_group(col):
    if '_club_' in col or col in ('neutral_venue', 'stage_league', 'stage_knockout'):
        return 'Club / Context'
    for pos in ['gk', 'def', 'mid', 'att']:
        if f'_{pos}_' in col or col.startswith(f'{pos}_'):
            return pos.upper()
    if '_bench_' in col or '_xi_' in col:
        return 'Squad Aggregates'
    return 'Other'


In [ ]:
if RUN_SHAP:
    global_shap(data, groups, saved, OUTPUT / 'global_shap')
    display(pd.read_csv(OUTPUT / 'global_shap_all_features.csv').head(15))

## 6. Local SHAP for candidate matches

The 2024–25 examples are part of refit training, while 2025–26 examples are in the test season. Interpret them separately.

In [ ]:
def find_cases(data):
    selected = []
    for title, season, date, home, away in CASE_REQUESTS:
        rows = data.loc[
            data.season.eq(season) & data.match_date.astype(str).str.startswith(date) &
            data.home_team.str.contains(home, case=False, regex=False, na=False) &
            data.away_team.str.contains(away, case=False, regex=False, na=False)
        ]
        if len(rows) == 1:
            row = rows.iloc[0]
            selected.append((title, row.name))
        else:
            print(f'Case unavailable or ambiguous ({len(rows)} rows): {title}')
    return selected


def local_shap(data, groups, saved, out):
    out.mkdir(parents=True, exist_ok=True)
    cases = find_cases(data)
    rows = []
    for title, index in cases:
        case = data.loc[index]
        for route, bundle in saved.items():
            X_all = numeric_input(bundle, groups[bundle['feature_set']])
            X = X_all.loc[[index]]
            bg = X_all.loc[data.season != '2025-26'].sample(n=min(100, (data.season != '2025-26').sum()), random_state=42)
            probs, raw = predict_bundle(bundle, X)
            label = decision_labels(route, probs, raw, bundle.get('threshold'))[0]
            rows.append({'case': title, 'match_id': case.match_id, 'season': case.season,
                         'data_split': 'test' if case.season == '2025-26' else 'train+valid',
                         'route': route, 'actual': case.label_90min_wdl,
                         'predicted': label, 'prob_A': probs[0, 0], 'prob_D': probs[0, 1], 'prob_H': probs[0, 2]})
            names = ['Home goals', 'Away goals'] if route == 'Two Goals' else ['WDL' if route == 'WDL' else 'Goal Difference']
            models = [bundle['models']['home'], bundle['models']['away']] if route == 'Two Goals' else [bundle['models']['main']]

            for name, model in zip(names, models):
                try:
                    sv = shap_for_pipeline(model, X, bg)
                    if sv.values.ndim == 3:
                        # Explain the predicted WDL class in the model-margin (logit) space.
                        class_index = int(np.argmax(probs[0]))
                        explanation = sv[0, :, class_index]
                        suffix = ORDER[class_index]
                    else:
                        explanation = sv[0]
                        suffix = name
                    plt.figure(figsize=(9, 5))
                    shap.plots.waterfall(explanation, max_display=15, show=False)
                    plt.title(f'{title}\n{route} | {suffix}')
                    safe = ''.join(ch if ch.isalnum() else '_' for ch in title + '_' + route + '_' + suffix)
                    plt.tight_layout()
                    plt.savefig(out / f'{safe}.png', dpi=180, bbox_inches='tight')
                    plt.close()
                except Exception as e:
                    print(f'Local SHAP failed: {title} / {route}: {e}')

    cases_df = pd.DataFrame(rows)
    cases_df.to_csv(out / 'case_predictions.csv', index=False)
    return cases_df


In [ ]:
if RUN_SHAP:
    local_shap(data, groups, saved, OUTPUT / 'local_shap')
    cases_file = OUTPUT / 'local_shap' / 'case_predictions.csv'
    if cases_file.exists():
        display(pd.read_csv(cases_file))

## 7. Interpretation notes

Retain all case outputs first; after reviewing them, select 2–3 informative matches for the report.

In [ ]:
def write_interpretation_notes(output):
    """Summarize feature evidence without treating SHAP as a causal explanation."""
    metrics_path = output / 'final_test_metrics.csv'
    group_path = output / 'global_shap' / 'global_shap_position_groups.csv'
    feature_path = output / 'global_shap' / 'global_shap_all_features.csv'
    cases_path = output / 'local_shap' / 'case_predictions.csv'

    lines = [
        '# Final Model Interpretation — Review Notes',
        '',
        'Models were chosen on rolling historical seasons, refitted on 2014–15 through 2024–25,',
        'and evaluated on 2025–26. Only player-containing configurations were eligible',
        'for final selection; Club Only remains a comparison baseline.',
        '',
        '## Prior research findings retained',
        '',
        '- Club ratings were consistently strong and high-dimensional individual player models overfit.',
        '- DEF interceptions, reactions, short passing, and sprint speed were promising;',
        '  MID reactions and short passing and ATT reactions and finishing also appeared repeatedly.',
        '- Player aggregates showed some predictive value, but their improvement beyond Club Only',
        '  was inconsistent across seasons and algorithms.',
        '- Residualized player features were excluded here to keep interpretation in original rating units.',
        '',
        '## Test performance',
        ''
    ]
    if metrics_path.exists():
        m = pd.read_csv(metrics_path)
        for _, row in m.iterrows():
            lines.append(f"- {row.route}: {row.feature_set} / {row.model}, "
                         f"Accuracy={row.accuracy:.3f}, Macro-AUC={row.macro_auc:.3f}, "
                         f"Macro-F1={row.macro_f1:.3f}.")

    lines.extend(['', '## Global SHAP findings', ''])
    if group_path.exists():
        g = pd.read_csv(group_path)
        for (route, label), sub in g.groupby(['route', 'output']):
            ranked = sub.sort_values('mean_abs_shap', ascending=False)
            desc = ', '.join(f"{r.group} ({r.mean_abs_shap:.3f})" for _, r in ranked.iterrows())
            lines.append(f'- {route} / {label}: {desc}.')
    if feature_path.exists():
        f = pd.read_csv(feature_path)
        lines.extend(['', '### Top individual features by model output', ''])
        for (route, label), sub in f.groupby(['route', 'output']):
            names = ', '.join(sub.nlargest(5, 'mean_abs_shap').feature)
            lines.append(f'- {route} / {label}: {names}.')

    lines.extend(['', '## Local cases', ''])
    if cases_path.exists():
        c = pd.read_csv(cases_path)
        for title, sub in c.groupby('case', sort=False):
            row = sub.iloc[0]
            lines.append(f'- {title} ({row.season}; {row.data_split}): observed {row.actual}. '
                         'Review the separate model waterfall plots and prediction probabilities.')

    lines.extend([
        '',
        '**Interpretation guardrails:** SHAP explains a fitted model output, not the actual',
        'cause of a match result. For WDL linear models, values describe class margins/logits;',
        'for Goal Difference they are goal-difference units; for Poisson they are log expected goals.',
        'These magnitudes are not directly comparable across the three routes.',
        '2024–25 matches are in the refit training period and are not independent test cases.',
        'The 2025–26 test season has been viewed during previous research iterations,',
        'so the final evaluation is not a completely untouched blind test.',
    ])
    (output / 'feature_value_report.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')


In [ ]:
if RUN_SHAP:
    write_interpretation_notes(OUTPUT)

print('Results saved to:', OUTPUT.resolve())